# Analista de datos con IA generativa y Spark
### Proyecto integrador — Computación en la Nube · Maestría en Ciencia de Datos

En este notebook se construye un **asistente de análisis de datos** sobre el archivo real `SistemadeAlertas.xlsx`. El usuario hace una pregunta en lenguaje natural y un modelo de lenguaje, GLM de Z.ai, **escribe el código de análisis en PySpark**. Luego ese código se ejecuta sobre el DataFrame `df` y el resultado se presenta en una aplicación web.

El objetivo del proyecto es integrar cuatro componentes: procesamiento distribuido con Spark, generación de código con un LLM, interfaz web con Gradio y despliegue en Hugging Face Spaces mediante Docker.

| Herramienta | Qué es | Para qué se usa aquí |
|---|---|---|
| **PySpark** | Motor de procesamiento de Big Data | Cargar, tipar y analizar el archivo real del Sistema de Alertas |
| **GLM de Z.ai** | Modelo de lenguaje vía API compatible con OpenAI | Traducir preguntas en lenguaje natural a código PySpark |
| **Gradio** | Interfaz web | Permitir que el usuario pregunte, vea el código generado y revise los resultados |
| **Hugging Face Spaces** | Plataforma de despliegue | Publicar la aplicación como contenedor Docker |

> Dataset usado: `SistemadeAlertas.xlsx`, hoja `Data General`.

## Paso 0 — Instalación de dependencias

In [ ]:
# PASO 0 — Instalación de dependencias (versiones fijas y compatibles entre sí).
#   pyspark          -> motor de procesamiento de datos
#   gradio           -> interfaz web
#   openai           -> cliente para llamar al modelo GLM de Z.ai (API compatible con OpenAI)
#   huggingface_hub  -> para publicar la app en Hugging Face
#   openpyxl         -> motor de lectura del archivo Excel real del Sistema de Alertas
!pip install -q   pyspark==4.0.0   gradio==6.17.3   openai   starlette==1.3.0   fastapi==0.136.3   huggingface_hub==1.19.0   openpyxl

print("Dependencias instaladas")

In [ ]:
# Verificación de versiones
# Sirve para confirmar, después de instalar y reiniciar la sesión, que quedaron
# las versiones correctas de starlette y gradio. Si no coinciden, la aplicación
# (celda 4b) puede fallar al iniciarse.
import starlette, gradio
print("starlette", starlette.__version__, "| gradio", gradio.__version__)
# Resultado esperado: starlette 1.3.0 | gradio 6.17.3

## Paso 1 — Spark y los datos reales del Sistema de Alertas
Se inicia Spark y se carga el archivo `SistemadeAlertas.xlsx`, que debe estar en el mismo entorno de Colab donde se ejecuta el notebook.

El resultado de este paso es el insumo principal del asistente: el LLM recibirá el esquema de `df` y escribirá código PySpark sobre estas columnas reales.

In [ ]:
# PASO 1 — Iniciar Spark
# Spark es el motor que procesa los datos. Aquí se crea la "sesión de Spark",
# que es el punto de entrada para trabajar con DataFrames.

import os
from pathlib import Path
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.sql import types as T

# Se construye la sesión de Spark:
#   appName("analista-alertas") -> nombre de la aplicación
#   master("local[*]")          -> ejecuta en esta misma máquina usando todos sus núcleos
#   spark.ui.enabled = false    -> desactiva el panel web de Spark en Colab
#   getOrCreate()               -> crea la sesión o reutiliza una si ya existe
spark = (
    SparkSession.builder
    .appName("analista-alertas")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

# Se baja el nivel de mensajes: solo se muestran errores, no las advertencias de Spark.
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# PASO 1 (continuación) — Cargar el Excel real y construir el DataFrame de Spark
# Archivo esperado: debe estar cargado en el mismo entorno de Colab con este nombre.
# En Colab normalmente queda disponible como /content/SistemadeAlertas.xlsx si se sube desde el panel de archivos.

ARCHIVO_XLSX = Path(os.environ.get("ALERTAS_XLSX", "SistemadeAlertas.xlsx"))
HOJA_XLSX = os.environ.get("ALERTAS_HOJA", "Data General")

COLUMNAS_RENOMBRADAS = {
    "id": "id",
    "Tipo de Alerta": "tipo_alerta",
    "País": "pais",
    "Fecha nacimiento": "fecha_nacimiento",
    "Genero": "genero",
    "Víctima del Conflicto armado": "victima_conflicto_armado",
    "Discapacidad": "discapacidad",
    "Etnia": "etnia",
    "Estrato": "estrato",
    "Tipo colegio": "tipo_colegio",
    "Jornada": "jornada",
    "Grado": "grado",
    "Localidad": "localidad",
    "UPZ": "upz",
    "Fecha": "fecha_alerta",
    "FechaNacReal": "fecha_nac_real",
    "FechaAlrtReal": "fecha_alrt_real",
    "Edad_caso": "edad_caso",
    "Edad_Real": "edad_real",
    "Nom_Loca": "localidad_nombre",
    "NivelAcad": "nivel_academico",
    "Año": "anio",
    "Semestre": "semestre",
    "Características diferenciales": "caracteristicas_diferenciales",
}

COLUMNAS_FECHA = ["fecha_nacimiento", "fecha_alerta", "fecha_nac_real", "fecha_alrt_real"]
COLUMNAS_ENTERAS = ["id", "estrato", "edad_caso", "edad_real", "anio", "semestre"]
COLUMNAS_TEXTO = [
    "tipo_alerta", "pais", "genero", "victima_conflicto_armado", "discapacidad",
    "etnia", "tipo_colegio", "jornada", "grado", "localidad", "upz",
    "localidad_nombre", "nivel_academico", "caracteristicas_diferenciales"
]

SCHEMA_ALERTAS = T.StructType([
    T.StructField("id", T.LongType(), True),
    T.StructField("tipo_alerta", T.StringType(), True),
    T.StructField("pais", T.StringType(), True),
    T.StructField("fecha_nacimiento", T.DateType(), True),
    T.StructField("genero", T.StringType(), True),
    T.StructField("victima_conflicto_armado", T.StringType(), True),
    T.StructField("discapacidad", T.StringType(), True),
    T.StructField("etnia", T.StringType(), True),
    T.StructField("estrato", T.IntegerType(), True),
    T.StructField("tipo_colegio", T.StringType(), True),
    T.StructField("jornada", T.StringType(), True),
    T.StructField("grado", T.StringType(), True),
    T.StructField("localidad", T.StringType(), True),
    T.StructField("upz", T.StringType(), True),
    T.StructField("fecha_alerta", T.DateType(), True),
    T.StructField("fecha_nac_real", T.DateType(), True),
    T.StructField("fecha_alrt_real", T.DateType(), True),
    T.StructField("edad_caso", T.IntegerType(), True),
    T.StructField("edad_real", T.IntegerType(), True),
    T.StructField("localidad_nombre", T.StringType(), True),
    T.StructField("nivel_academico", T.StringType(), True),
    T.StructField("anio", T.IntegerType(), True),
    T.StructField("semestre", T.IntegerType(), True),
    T.StructField("caracteristicas_diferenciales", T.StringType(), True),
])


def _convertir_fecha_excel(serie: pd.Series) -> pd.Series:
    """Convierte columnas de fecha leídas desde Excel a objetos date o None."""
    if pd.api.types.is_datetime64_any_dtype(serie):
        fechas = pd.to_datetime(serie, errors="coerce")
    else:
        serie_num = pd.to_numeric(serie, errors="coerce")
        proporcion_numerica = serie_num.notna().mean()
        if proporcion_numerica > 0.80:
            fechas = pd.to_datetime(serie_num, unit="D", origin="1899-12-30", errors="coerce")
        else:
            fechas = pd.to_datetime(serie, errors="coerce")

    return fechas.apply(lambda x: None if pd.isna(x) else x.date())


def _entero_nullable(serie: pd.Series) -> pd.Series:
    """Convierte una columna a enteros de Python o None, compatible con Spark + esquema explícito."""
    serie_num = pd.to_numeric(serie, errors="coerce")
    return pd.Series([int(x) if pd.notna(x) else None for x in serie_num], index=serie.index, dtype="object")


def _texto_nullable(serie: pd.Series) -> pd.Series:
    """Convierte valores de texto a string limpio o None."""
    def limpiar(x):
        if pd.isna(x):
            return None
        valor = str(x).strip()
        return valor if valor else None
    return serie.apply(limpiar)


def cargar_alertas_excel(ruta_excel: Path = ARCHIVO_XLSX, hoja: str = HOJA_XLSX):
    """Lee el Excel real, normaliza columnas, aplica tipos y devuelve un DataFrame de Spark."""
    if not ruta_excel.exists():
        raise FileNotFoundError(
            f"No se encontró {ruta_excel}. Sube el archivo SistemadeAlertas.xlsx al entorno de Colab."
        )

    pdf = pd.read_excel(ruta_excel, sheet_name=hoja, engine="openpyxl")
    pdf = pdf.rename(columns=COLUMNAS_RENOMBRADAS)

    columnas_esperadas = list(COLUMNAS_RENOMBRADAS.values())
    faltantes = [c for c in columnas_esperadas if c not in pdf.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas esperadas en el Excel: {faltantes}")

    pdf = pdf[columnas_esperadas].copy()

    for columna in COLUMNAS_FECHA:
        pdf[columna] = _convertir_fecha_excel(pdf[columna])

    for columna in COLUMNAS_ENTERAS:
        pdf[columna] = _entero_nullable(pdf[columna])

    for columna in COLUMNAS_TEXTO:
        pdf[columna] = _texto_nullable(pdf[columna])

    df_spark = spark.createDataFrame(pdf, schema=SCHEMA_ALERTAS)

    # Variables derivadas útiles para preguntas analíticas frecuentes.
    df_spark = (
        df_spark
        .withColumn("mes_alerta", F.month("fecha_alerta"))
        .withColumn("trimestre_alerta", F.quarter("fecha_alerta"))
    )

    return df_spark


In [ ]:
df = cargar_alertas_excel()
ESQUEMA = ", ".join([f"{c} ({t})" for c, t in df.dtypes])

print("Filas:", df.count(), "| Columnas:", len(df.columns))
print("Esquema:", ESQUEMA)
df.show(5, truncate=False)

## Paso 2 — El LLM que escribe el código (GLM de Z.ai)

El LLM recibe la pregunta y el esquema de los datos, y devuelve el código PySpark que la responde.

Se necesita una **API key gratuita** de Z.ai (no pide tarjeta de crédito):

1. Entrar a **z.ai** e iniciar sesión (con correo o cuenta de Google).
2. Ir a la página **API Keys** y crear una clave con **"Create API key"**.
3. Copiar la clave (formato `{API Key ID}.{secret}`, con un punto en medio) y pegarla cuando la celda la solicite.

> El detalle paso a paso está en el **Manual_ZAI_API_Key**.

In [ ]:
# PASO 2 — Conectar el modelo de lenguaje (GLM de Z.ai)
# Librerías necesarias para este paso:
from getpass import getpass
from openai import OpenAI

# Clave de Z.ai. Se ingresa de forma segura y queda en una variable de entorno temporal de Colab.
os.environ["ZAI_API_KEY"] = getpass("API key de Z.ai (GLM): ")

# Z.ai es compatible con OpenAI: se usa el cliente de OpenAI apuntando al servidor de Z.ai.
cliente = OpenAI(api_key=os.environ["ZAI_API_KEY"], base_url="https://api.z.ai/api/paas/v4")
MODELO_LLM = "glm-4.5-flash"

# Instrucciones para que el modelo devuelva SOLO código PySpark adaptado al Sistema de Alertas.
INSTRUCCION = f"""Eres un analista de datos experto en PySpark.
Existe un DataFrame de Spark llamado df con datos reales del Sistema de Alertas.
Columnas disponibles: {ESQUEMA}.
Escribe código PySpark que responda la pregunta del usuario.
Reglas estrictas:
- Usa únicamente df, spark y F (pyspark.sql.functions). No leas ni escribas archivos.
- Usa exactamente los nombres de columnas disponibles en el esquema.
- Para conteos usa F.count("*") y para promedios usa F.avg().
- Asigna SIEMPRE el resultado final a una variable llamada resultado.
- Si el resultado es una tabla, deja resultado como un DataFrame de Spark y no uses .show().
- Devuelve SOLO el código Python, sin explicaciones ni bloques de markdown."""

def generar_codigo(pregunta):
    """Pide al modelo GLM el código PySpark para la pregunta y limpia la respuesta."""
    r = cliente.chat.completions.create(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": INSTRUCCION},
            {"role": "user", "content": pregunta},
        ],
    )
    codigo = r.choices[0].message.content.strip()
    if codigo.startswith("```"):
        codigo = codigo.strip("`")
        if codigo.lstrip().startswith("python"):
            codigo = codigo.lstrip()[len("python"):]
    return codigo.strip()

# Prueba: se le pide el código para una pregunta propia del Sistema de Alertas.
print(generar_codigo("¿Cuántas alertas hay por tipo_alerta?"))

## Paso 3 — Ejecutar el código generado
El código que escribió el LLM se ejecuta sobre los datos. **Importante:** se ejecuta código generado por la IA; por eso corre en un entorno controlado, solo con `df`, `spark` y `F`, y sobre datos propios.

## Paso 4 — La aplicación (interfaz web)
Se guarda toda la lógica en `app.py`: Spark, la carga del Excel real, el LLM GLM de Z.ai, la ejecución controlada del código generado y la interfaz de Gradio. La API key se lee de la variable de entorno `ZAI_API_KEY`; en Hugging Face debe configurarse como un **secret**.

In [ ]:
%%writefile app.py
# Esta celda crea el archivo app.py: la aplicación que luego se despliega en Hugging Face.
# (%%writefile guarda en un archivo todo lo que está debajo; no lo ejecuta aquí.)
"""
Analista de datos con IA generativa y Spark sobre el Sistema de Alertas.
El usuario escribe una pregunta en lenguaje natural; el modelo GLM de Z.ai traduce esa
pregunta a código PySpark, el código se ejecuta sobre los datos reales y se muestra el resultado.
La interfaz es web con Gradio. La clave de Z.ai se lee desde la variable de entorno ZAI_API_KEY.
"""
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import gradio as gr
from openai import OpenAI
from pyspark.sql import SparkSession, functions as F, types as T

# ---------- 1) Iniciar Spark ----------
spark = (
    SparkSession.builder
    .appName("analista-alertas")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# ---------- 1b) Cargar datos reales desde Excel ----------
# En Hugging Face el archivo viaja dentro del contenedor porque el Dockerfile lo copia a /app.
# En Colab o ejecución local, el archivo debe estar en el mismo directorio o se puede definir ALERTAS_XLSX.
ARCHIVO_XLSX = Path(os.environ.get("ALERTAS_XLSX", "SistemadeAlertas.xlsx"))
HOJA_XLSX = os.environ.get("ALERTAS_HOJA", "Data General")

COLUMNAS_RENOMBRADAS = {
    "id": "id",
    "Tipo de Alerta": "tipo_alerta",
    "País": "pais",
    "Fecha nacimiento": "fecha_nacimiento",
    "Genero": "genero",
    "Víctima del Conflicto armado": "victima_conflicto_armado",
    "Discapacidad": "discapacidad",
    "Etnia": "etnia",
    "Estrato": "estrato",
    "Tipo colegio": "tipo_colegio",
    "Jornada": "jornada",
    "Grado": "grado",
    "Localidad": "localidad",
    "UPZ": "upz",
    "Fecha": "fecha_alerta",
    "FechaNacReal": "fecha_nac_real",
    "FechaAlrtReal": "fecha_alrt_real",
    "Edad_caso": "edad_caso",
    "Edad_Real": "edad_real",
    "Nom_Loca": "localidad_nombre",
    "NivelAcad": "nivel_academico",
    "Año": "anio",
    "Semestre": "semestre",
    "Características diferenciales": "caracteristicas_diferenciales",
}

COLUMNAS_FECHA = ["fecha_nacimiento", "fecha_alerta", "fecha_nac_real", "fecha_alrt_real"]
COLUMNAS_ENTERAS = ["id", "estrato", "edad_caso", "edad_real", "anio", "semestre"]
COLUMNAS_TEXTO = [
    "tipo_alerta", "pais", "genero", "victima_conflicto_armado", "discapacidad",
    "etnia", "tipo_colegio", "jornada", "grado", "localidad", "upz",
    "localidad_nombre", "nivel_academico", "caracteristicas_diferenciales"
]

SCHEMA_ALERTAS = T.StructType([
    T.StructField("id", T.LongType(), True),
    T.StructField("tipo_alerta", T.StringType(), True),
    T.StructField("pais", T.StringType(), True),
    T.StructField("fecha_nacimiento", T.DateType(), True),
    T.StructField("genero", T.StringType(), True),
    T.StructField("victima_conflicto_armado", T.StringType(), True),
    T.StructField("discapacidad", T.StringType(), True),
    T.StructField("etnia", T.StringType(), True),
    T.StructField("estrato", T.IntegerType(), True),
    T.StructField("tipo_colegio", T.StringType(), True),
    T.StructField("jornada", T.StringType(), True),
    T.StructField("grado", T.StringType(), True),
    T.StructField("localidad", T.StringType(), True),
    T.StructField("upz", T.StringType(), True),
    T.StructField("fecha_alerta", T.DateType(), True),
    T.StructField("fecha_nac_real", T.DateType(), True),
    T.StructField("fecha_alrt_real", T.DateType(), True),
    T.StructField("edad_caso", T.IntegerType(), True),
    T.StructField("edad_real", T.IntegerType(), True),
    T.StructField("localidad_nombre", T.StringType(), True),
    T.StructField("nivel_academico", T.StringType(), True),
    T.StructField("anio", T.IntegerType(), True),
    T.StructField("semestre", T.IntegerType(), True),
    T.StructField("caracteristicas_diferenciales", T.StringType(), True),
])


def _convertir_fecha_excel(serie: pd.Series) -> pd.Series:
    """Convierte columnas de fecha leídas desde Excel a objetos date o None."""
    if pd.api.types.is_datetime64_any_dtype(serie):
        fechas = pd.to_datetime(serie, errors="coerce")
    else:
        serie_num = pd.to_numeric(serie, errors="coerce")
        proporcion_numerica = serie_num.notna().mean()
        if proporcion_numerica > 0.80:
            fechas = pd.to_datetime(serie_num, unit="D", origin="1899-12-30", errors="coerce")
        else:
            fechas = pd.to_datetime(serie, errors="coerce")

    return fechas.apply(lambda x: None if pd.isna(x) else x.date())


def _entero_nullable(serie: pd.Series) -> pd.Series:
    serie_num = pd.to_numeric(serie, errors="coerce")
    return pd.Series([int(x) if pd.notna(x) else None for x in serie_num], index=serie.index, dtype="object")


def _texto_nullable(serie: pd.Series) -> pd.Series:
    def limpiar(x):
        if pd.isna(x):
            return None
        valor = str(x).strip()
        return valor if valor else None
    return serie.apply(limpiar)


def cargar_alertas_excel(ruta_excel: Path = ARCHIVO_XLSX, hoja: str = HOJA_XLSX):
    """Lee el Excel real, normaliza columnas, aplica tipos y devuelve un DataFrame de Spark."""
    if not ruta_excel.exists():
        raise FileNotFoundError(
            f"No se encontró {ruta_excel}. El archivo debe subirse al Space o al entorno local."
        )

    pdf = pd.read_excel(ruta_excel, sheet_name=hoja, engine="openpyxl")
    pdf = pdf.rename(columns=COLUMNAS_RENOMBRADAS)

    columnas_esperadas = list(COLUMNAS_RENOMBRADAS.values())
    faltantes = [c for c in columnas_esperadas if c not in pdf.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas esperadas en el Excel: {faltantes}")

    pdf = pdf[columnas_esperadas].copy()

    for columna in COLUMNAS_FECHA:
        pdf[columna] = _convertir_fecha_excel(pdf[columna])
    for columna in COLUMNAS_ENTERAS:
        pdf[columna] = _entero_nullable(pdf[columna])
    for columna in COLUMNAS_TEXTO:
        pdf[columna] = _texto_nullable(pdf[columna])

    df_spark = spark.createDataFrame(pdf, schema=SCHEMA_ALERTAS)
    df_spark = (
        df_spark
        .withColumn("mes_alerta", F.month("fecha_alerta"))
        .withColumn("trimestre_alerta", F.quarter("fecha_alerta"))
    )
    return df_spark


df = cargar_alertas_excel()
ESQUEMA = ", ".join([f"{c} ({t})" for c, t in df.dtypes])

# ---------- 2) El LLM que escribe el código (GLM de Z.ai) ----------
MODELO_LLM = "glm-4.5-flash"

_cliente = None
def _get_cliente():
    """Crea el cliente OpenAI -> Z.ai la primera vez que se usa."""
    global _cliente
    if _cliente is None:
        api_key = os.environ.get("ZAI_API_KEY", "")
        if not api_key:
            raise RuntimeError("Falta la variable de entorno ZAI_API_KEY. Configúrala como Secret en Hugging Face.")
        _cliente = OpenAI(api_key=api_key, base_url="https://api.z.ai/api/paas/v4")
    return _cliente

INSTRUCCION = f"""Eres un analista de datos experto en PySpark.
Existe un DataFrame de Spark llamado df con datos reales del Sistema de Alertas.
Columnas disponibles: {ESQUEMA}.
Escribe código PySpark que responda la pregunta del usuario.
Reglas estrictas:
- Usa únicamente df, spark y F (pyspark.sql.functions). No leas ni escribas archivos.
- Usa exactamente los nombres de columnas disponibles en el esquema.
- Para conteos usa F.count("*") y para promedios usa F.avg().
- Asigna SIEMPRE el resultado final a una variable llamada resultado.
- Si el resultado es una tabla, deja resultado como un DataFrame de Spark y no uses .show().
- No importes librerías, no uses os, no uses open, no leas ni escribas archivos.
- Devuelve SOLO el código Python, sin explicaciones ni bloques de markdown."""


def generar_codigo(pregunta):
    """Pide al modelo GLM el código PySpark para la pregunta y limpia la respuesta."""
    r = _get_cliente().chat.completions.create(
        model=MODELO_LLM,
        messages=[
            {"role": "system", "content": INSTRUCCION},
            {"role": "user", "content": pregunta},
        ],
    )
    codigo = r.choices[0].message.content.strip()
    if codigo.startswith("```"):
        codigo = codigo.strip("`")
        if codigo.lstrip().startswith("python"):
            codigo = codigo.lstrip()[len("python"):]
    return codigo.strip()

# ---------- 3) Ejecutar el código generado ----------
PROHIBIDO = [
    "import ", "__", "open(", "exec(", "eval(", "subprocess", "os.", "sys.",
    "read.", ".read(", "write.", ".write(", "saveAs", "to_csv", "to_excel"
]


def validar_codigo(codigo: str):
    codigo_min = codigo.lower()
    encontrados = [patron for patron in PROHIBIDO if patron.lower() in codigo_min]
    if encontrados:
        raise ValueError(f"El código generado usa instrucciones no permitidas: {encontrados}")


def ejecutar(codigo):
    """Ejecuta el código en un entorno controlado y devuelve 'resultado' como tabla."""
    validar_codigo(codigo)
    entorno = {"spark": spark, "df": df, "F": F, "resultado": None}
    exec(codigo, {"__builtins__": {}}, entorno)
    res = entorno["resultado"]
    if res is None:
        return None
    if hasattr(res, "toPandas"):
        return res.limit(50).toPandas()
    return pd.DataFrame({"resultado": [res]})

# ---------- 4) Unir todo: pregunta -> código -> resultado ----------
def analizar(pregunta):
    if not pregunta or not pregunta.strip():
        return "", None, None
    try:
        codigo = generar_codigo(pregunta)
    except Exception as e:
        return f"# Error al generar el código: {e}", None, None
    try:
        tabla = ejecutar(codigo)
    except Exception as e:
        return codigo, pd.DataFrame({"Error": [str(e)]}), None

    fig = None
    if tabla is not None and tabla.shape[1] == 2 and pd.api.types.is_numeric_dtype(tabla.iloc[:, 1]):
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.bar(tabla.iloc[:, 0].astype(str), tabla.iloc[:, 1])
        ax.set_ylabel(tabla.columns[1])
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
    return codigo, tabla, fig

# ---------- 5) Interfaz web ----------
EJEMPLOS = [
    "¿Cuántas alertas hay por tipo_alerta?",
    "Muestra las 10 localidades con más alertas en 2025",
    "¿Cuál es la distribución de alertas por genero?",
    "Calcula el promedio de edad_caso por nivel_academico",
    "Compara el número de alertas por anio y semestre",
]

with gr.Blocks(title="Analista de datos con IA - Sistema de Alertas") as demo:
    gr.Markdown(
        "# Analista de datos con IA y Spark\n"
        "Escribe una pregunta en lenguaje natural. La IA escribe el análisis en PySpark, "
        "se ejecuta sobre el Sistema de Alertas y se muestra el resultado."
    )
    pregunta = gr.Textbox(label="Tu pregunta", placeholder="¿Cuántas alertas hay por localidad_nombre?")
    btn = gr.Button("Analizar", variant="primary")
    gr.Examples(EJEMPLOS, inputs=pregunta)
    codigo_out = gr.Code(label="Código PySpark que escribió la IA", language="python")
    tabla_out = gr.Dataframe(label="Resultado")
    chart_out = gr.Plot(label="Gráfico, si aplica")
    btn.click(analizar, inputs=pregunta, outputs=[codigo_out, tabla_out, chart_out])

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860)


## Paso 4b — Prueba local de la aplicación
Se ejecuta la app dentro de Colab para verla funcionar. `share=True` genera un enlace público temporal. (Para continuar, detener la celda.)

In [ ]:
import importlib, app as app_module
importlib.reload(app_module)
app_module.demo.launch(share=True)

## Paso 5 — Empaquetado para el despliegue

In [ ]:
import pyspark, gradio, pandas, numpy, matplotlib, openai, openpyxl, fastapi, starlette

# Se anotan las versiones exactas para que el contenedor reproduzca este entorno.
reqs = (
    f"pyspark=={pyspark.__version__}\n"
    f"gradio=={gradio.__version__}\n"
    f"openai=={openai.__version__}\n"
    f"pandas=={pandas.__version__}\n"
    f"numpy=={numpy.__version__}\n"
    f"matplotlib=={matplotlib.__version__}\n"
    f"openpyxl=={openpyxl.__version__}\n"
    f"fastapi=={fastapi.__version__}\n"
    f"starlette=={starlette.__version__}\n"
)
open("requirements.txt", "w").write(reqs)
print(reqs)

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

# Spark necesita Java: se instala el kit de desarrollo de Java.
RUN apt-get update && apt-get install -y default-jdk && rm -rf /var/lib/apt/lists/*

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY SistemadeAlertas.xlsx .

EXPOSE 7860
CMD ["python", "app.py"]

In [ ]:
%%writefile README.md
---
title: Analista de datos con IA - Sistema de Alertas -- Adrian Grateron - Jhonatan Smith Doncell - Jorge Emmanuel Marin
emoji: 🤖
colorFrom: yellow
colorTo: indigo
sdk: docker
app_port: 7860
pinned: false
---
# Analista de datos con IA y Spark

Aplicación que recibe preguntas en lenguaje natural, usa el modelo GLM de Z.ai para generar código PySpark y ejecuta el análisis sobre el archivo real `SistemadeAlertas.xlsx`.

## Datos

- Fuente usada por la aplicación: `SistemadeAlertas.xlsx`
- Hoja: `Data General`
- DataFrame principal: `df`
- Variables derivadas: `mes_alerta` y `trimestre_alerta`

## Ejemplos de preguntas

- ¿Cuántas alertas hay por tipo_alerta?
- Muestra las 10 localidades con más alertas en 2025
- ¿Cuál es la distribución de alertas por genero?
- Calcula el promedio de edad_caso por nivel_academico
- Compara el número de alertas por anio y semestre

## Credenciales

La clave de Z.ai debe configurarse como Secret del Space con el nombre `ZAI_API_KEY`.

## Informe:



## Paso 6 — Despliegue en Hugging Face Spaces
Igual que en la clase de despliegue, pero con dos ajustes para este proyecto: la API key de Z.ai se sube como **secret** del Space y el archivo `SistemadeAlertas.xlsx` se sube junto con `app.py`, `requirements.txt`, `Dockerfile` y `README.md`. El contenedor instala Java + Spark, por lo que la construcción puede tardar algunos minutos.

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi, create_repo, add_space_secret

HF_TOKEN = getpass("Token de Hugging Face (Write): ")
USUARIO  = input("Usuario de Hugging Face: ").strip()
SPACE_ID = f"{USUARIO}/analista-datos-ia-alertas"

create_repo(SPACE_ID, repo_type="space", space_sdk="docker", token=HF_TOKEN, exist_ok=True)
api = HfApi(token=HF_TOKEN)

# Se guarda la clave de Z.ai como secret del Space. La app la lee desde ZAI_API_KEY.
add_space_secret(SPACE_ID, "ZAI_API_KEY", os.environ["ZAI_API_KEY"], token=HF_TOKEN)

for archivo in ["app.py", "requirements.txt", "Dockerfile", "README.md", "SistemadeAlertas.xlsx"]:
    api.upload_file(path_or_fileobj=archivo, path_in_repo=archivo, repo_id=SPACE_ID, repo_type="space")
    print("subido:", archivo)

print("\nDesplegando. La construcción con Java + Spark puede tardar varios minutos.")
print(f"Estado del Space: https://huggingface.co/spaces/{SPACE_ID}")

---
## Cierre del proyecto

Esta aplicación reúne el flujo completo del proyecto integrador:

- **Spark**: carga el archivo real del Sistema de Alertas, aplica esquema explícito y ejecuta operaciones distribuidas.
- **IA generativa**: el LLM convierte preguntas en lenguaje natural en código PySpark ajustado al esquema del dataset.
- **Interfaz web**: Gradio permite hacer preguntas, revisar el código generado, observar la tabla resultante y visualizar un gráfico cuando aplica.
- **Despliegue**: Hugging Face Spaces construye el contenedor Docker, copia el Excel real y ejecuta la aplicación en el puerto 7860.

### Entrega
1. Ejecutar el notebook completo en Colab con `SistemadeAlertas.xlsx` cargado en el mismo entorno.
2. Probar preguntas de análisis sobre `tipo_alerta`, `localidad_nombre`, `genero`, `nivel_academico`, `anio`, `semestre`, `mes_alerta` y `trimestre_alerta`.
3. Desplegar en Hugging Face y entregar la URL pública del Space.

### Nota técnica
El mismo patrón puede escalar a un clúster real: la IA genera código PySpark sobre el DataFrame `df`, por lo que el diseño es transferible a Databricks u otro entorno Spark distribuido.